# Phase 1: Legal Encoder–Decoder Pretraining with the Project BPE Tokenizer

The Encoder–Decoder model and its training objective are retained. **Corpus acquisition, reservoir sampling, concatenation, and data cleaning are aligned with the Transformer-XL notebook** so both models use the same preprocessing policy and sampling caps.

The custom project `BPETokenizer` is retained for encoding the cleaned corpus.


## 0. Environment, imports, and configuration

In [ ]:
# Colab: uncomment if needed
!pip -q install ijson huggingface_hub

import os, re, json, math, time, random, hashlib, gc
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/Phase1_Pretraining"

WORK_DIR = os.path.join(
    DRIVE_ROOT,
    "phase1_encoder_decoder"
)

CHECKPOINT_DIR = os.path.join(
    WORK_DIR,
    "checkpoints"
)

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TOKENIZER_PATH = os.path.join(WORK_DIR, "bpe_legal_32768.json")


# Start with a modest, GPU-friendly baseline. Scale only after the first run is stable.
VOCAB_SIZE = 32768
MAX_TRAIN_DOCS = 25000000       # set None to use every locally available document
VAL_FRAC = 0.05
MAX_ENC_LEN = 512            # BPE tokens, not words
MAX_DEC_LEN = 128            # target BPE tokens including EOS
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
EPOCHS = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
GRAD_CLIP = 1.0
LOG_EVERY = 25
VAL_EVERY = 200
SAVE_EVERY = 200
USE_AMP = (DEVICE.type == "cuda")
NUM_WORKERS = 0              # robust on Colab/notebook environments
MAX_BATCHES_PER_EPOCH = None # set e.g. 100 for a quick smoke test

os.environ['HF_TOKEN'] = "HF-TOKEN"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 5.5 MB/s eta 0:00:00
Device: cuda
GPU: Tesla T4
Mounted at /content/drive


## 1. Acquire and sample the Phase-1 legal corpus

This follows the data acquisition pattern in the Transformer-XL notebook:
- download the four `Phase_1_Pretraining/Train/*.json` files from `Exploration-Lab/CS-781-Capstone`;
- stream each JSON array with `ijson`;
- reservoir-sample the same per-source caps used in the Transformer-XL main preprocessing path;
- concatenate and shuffle the samples before cleaning.

Set `HF_TOKEN` in the notebook environment if Hugging Face authentication is required. No access token is stored in this notebook.


In [ ]:
from huggingface_hub import hf_hub_download
import os

# Use an environment variable for private/gated access:
# os.environ["HF_TOKEN"] = "..."  # set securely in your runtime; do not commit tokens
repo_id = "Exploration-Lab/CS-781-Capstone"
hf_token = os.environ.get("HF_TOKEN")

corpus_filenames = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

corpus_paths = {}
for source, filename in corpus_filenames.items():
    corpus_paths[source] = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        token=hf_token,
    )
    print(f"{source}: {corpus_paths[source]}")

# These are the same four Phase-1 JSON files used by the Transformer-XL notebook.


acts_path = corpus_paths["acts"]
high_courts_path = corpus_paths["high_courts"]
supreme_court_path = corpus_paths["supreme_court"]
tribunals_path = corpus_paths["tribunals"]


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

acts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_acts.json


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

high_courts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_high_courts.json


... (2 outputs left out)

supreme_court: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_supreme_court.json


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

tribunals: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_tribunals.json


In [ ]:
import ijson
import random
import json
import os


def reservoir_sample_json(
    path,
    n_samples,
    seed=42,
):
    rng = random.Random(seed)

    reservoir = []
    seen = 0

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            if not isinstance(record, dict):
                continue

            text = record.get("text")

            if not isinstance(text, str):
                continue

            seen += 1

            if len(reservoir) < n_samples:
                reservoir.append(text)

            else:
                j = rng.randint(0, seen - 1)

                if j < n_samples:
                    reservoir[j] = text

    print(
        f"{os.path.basename(path)}: "
        f"scanned {seen:,} documents, "
        f"selected {len(reservoir):,}"
    )

    return reservoir

In [ ]:
# Not needed for the JSON-array corpus files. The loader below streams records
# with ijson and samples them using reservoir sampling, matching Transformer-XL.


In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}-large.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=20_000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=300_000,
    seed=42,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=50_000,
    seed=42,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=300_000,
    seed=42,
)

train_acts.json: scanned 18,826 documents, selected 18,826
train_high_courts.json: scanned 2,536,738 documents, selected 300,000
train_supreme_court.json: scanned 55,549 documents, selected 50,000
train_tribunals.json: scanned 869,132 documents, selected 300,000


In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=20,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=300,
    seed=42,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=50,
    seed=42,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=300,
    seed=42,
)

train_acts.json: scanned 18,826 documents, selected 20
train_high_courts.json: scanned 2,536,738 documents, selected 300
train_supreme_court.json: scanned 55,549 documents, selected 50
train_tribunals.json: scanned 869,132 documents, selected 300


In [ ]:
import random

rng = random.Random(42)

# Concatenate the per-source reservoir samples into one document-level corpus.
# Each element is a document string, matching the Transformer-XL preprocessing pipeline.
train_val_texts = acts + high_courts + supreme_court + tribunals
# rng.shuffle(train_val_texts)

print(f"Total sampled documents before cleaning: {len(train_val_texts):,}")
print(f"Total characters before cleaning: {sum(map(len, train_val_texts)):,}")


Total sampled documents before cleaning: 668,826
Total characters before cleaning: 5,105,547,642


In [ ]:
import random

rng = random.Random(42)

# Concatenate the per-source reservoir samples into one document-level corpus.
# Each element is a document string, matching the Transformer-XL preprocessing pipeline.
train_val_texts = acts + high_courts + supreme_court + tribunals
# rng.shuffle(train_val_texts)

print(f"Total sampled documents before cleaning: {len(train_val_texts):,}")
print(f"Total characters before cleaning: {sum(map(len, train_val_texts)):,}")


Total sampled documents before cleaning: 670
Total characters before cleaning: 5,083,688


## 2. Clean and deduplicate documents

This uses the same `normalize_text`, English/script filtering, exact deduplication, and MinHash-LSH near-deduplication implementation and thresholds as the Transformer-XL notebook. Cleaning runs **before** the train/validation split to reduce duplicate leakage.


In [ ]:
# ============================================================
# DATA CLEANING
#   1. Unicode + whitespace normalisation
#   2. Language / script filtering (keep English, drop the rest)
#   3. De-duplication (exact + near-duplicate via MinHash-LSH)
#
# Run this AFTER the raw documents are sampled (train_val_texts)
# and BEFORE the train / val split, so that duplicates cannot
# leak from train into validation.
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text


# ------------------------------------------------------------
# 2. LANGUAGE / SCRIPT HANDLING (no external dependency)
# ------------------------------------------------------------
_LATIN = "A-Za-z\u00C0-\u024F\u1E00-\u1EFF"
_SCRIPTS = {
    "devanagari": "\u0900-\u097F", "bengali": "\u0980-\u09FF",
    "gurmukhi":   "\u0A00-\u0A7F", "gujarati": "\u0A80-\u0AFF",
    "odia":       "\u0B00-\u0B7F", "tamil":    "\u0B80-\u0BFF",
    "telugu":     "\u0C00-\u0C7F", "kannada":  "\u0C80-\u0CFF",
    "malayalam":  "\u0D00-\u0D7F",
    "arabic/urdu": "\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF",
    "cyrillic":   "\u0400-\u04FF",
    "cjk":        "\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF",
}
_SCRIPT_RES = {k: re.compile(f"[{v}]") for k, v in _SCRIPTS.items()}
_ALPHA_RE = re.compile(r"[^\W\d_]")
_LATIN_RE = re.compile(f"[{_LATIN}]")
# a run of non-Latin letters (plus the combining marks / joiners that belong to them)
_FOREIGN_RUN_RE = re.compile(
    "[" + "".join(_SCRIPTS.values()) + "]+"
    "(?:[\\s\u200c\u200d]*[" + "".join(_SCRIPTS.values()) + "]+)*"
)
_EN_WORD_RE = re.compile(r"[a-z]+")
_ENGLISH_STOP = frozenset("""
the of and to in that is was for by on with as be it at this which are from or not has have
had been an a he she they his her their its were any such under also but if all no can shall
may would
""".split())


def language_profile(text, sample_chars=3000):
    """Cheap profile from the start and middle of a document.
    Returns (latin_ratio, english_stopword_ratio, dominant_foreign_script)."""
    n = len(text)
    s = text if n <= 2 * sample_chars else (
        text[:sample_chars] + " " + text[n // 2: n // 2 + sample_chars])
    n_alpha = len(_ALPHA_RE.findall(s))
    if n_alpha == 0:
        return 0.0, 0.0, "none"
    n_latin = len(_LATIN_RE.findall(s))
    latin_ratio = n_latin / n_alpha

    words = _EN_WORD_RE.findall(s.lower())
    en_ratio = (sum(w in _ENGLISH_STOP for w in words) / len(words)) if words else 0.0

    foreign = "latin-non-english"
    if latin_ratio < 0.5:
        counts = {k: len(r.findall(s)) for k, r in _SCRIPT_RES.items()}
        best = max(counts, key=counts.get)
        foreign = best if counts[best] > 0 else "other"
    return latin_ratio, en_ratio, foreign


# ------------------------------------------------------------
# 3. NEAR-DUPLICATE DETECTION (MinHash + LSH banding)
# ------------------------------------------------------------
_WORD_RE = re.compile(r"\w+")
_MH_PRIME = np.uint64(4294967311)           # prime just above 2**32
_ROLL = np.uint64(1000003)


class NearDupIndex:
    """Streaming near-duplicate detector. The first copy seen is kept."""

    def __init__(self, num_perm=64, bands=8, threshold=0.8,
                 shingle=5, max_words=2000, seed=1):
        assert num_perm % bands == 0
        self.num_perm, self.bands = num_perm, bands
        self.rows = num_perm // bands
        self.threshold, self.shingle, self.max_words = threshold, shingle, max_words
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.b = rng.randint(0, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.sigs = []
        self.buckets = [dict() for _ in range(bands)]

    def _shingle_hashes(self, words):
        words = words[: self.max_words]
        k = self.shingle
        wh = np.fromiter((zlib.crc32(w.encode()) for w in words),
                         dtype=np.uint64, count=len(words))
        m = len(wh) - k + 1
        if m < 1:
            return wh
        sh = wh[:m].copy()
        for j in range(1, k):                 # rolling hash of k consecutive words
            sh = sh * _ROLL + wh[j: m + j]
        return (sh ^ (sh >> np.uint64(32))) & np.uint64(0xFFFFFFFF)

    def check_and_add(self, words):
        """True if a near-duplicate was already indexed; otherwise index and return False."""
        h = self._shingle_hashes(words)[None, :]
        sig = ((self.a * h + self.b) % _MH_PRIME).min(axis=1)
        r = self.rows
        keys = [hash(sig[i * r:(i + 1) * r].tobytes()) for i in range(self.bands)]
        for i, k in enumerate(keys):
            j = self.buckets[i].get(k)
            if j is not None and (self.sigs[j] == sig).mean() >= self.threshold:
                return True
        idx = len(self.sigs)
        self.sigs.append(sig)
        for i, k in enumerate(keys):
            self.buckets[i].setdefault(k, idx)
        return False


# ------------------------------------------------------------
# PIPELINE
# ------------------------------------------------------------
def clean_corpus(
    texts,
    min_words=30,                 # drop near-empty documents
    language_filter=True,         # drop non-English documents
    min_latin_ratio=0.90,         # share of letters that must be Latin script
    min_english_ratio=0.12,       # share of words that are common English words
    strip_foreign_runs=True,      # remove stray Hindi/Urdu/... fragments inside English docs
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,          # estimated Jaccard similarity above which = duplicate
    collapse_repeats=True,
    n_examples=3,
    progress_every=20_000,
):
    stats = collections.Counter()
    dropped_scripts = collections.Counter()
    examples = collections.defaultdict(list)
    seen_exact = set()
    near = NearDupIndex(threshold=near_threshold) if near_dedup else None
    kept = []

    def drop(reason, text):
        stats[reason] += 1
        if len(examples[reason]) < n_examples:
            examples[reason].append(text[:200])

    for n, raw in enumerate(texts, 1):
        stats["input_docs"] += 1
        if not isinstance(raw, str):
            drop("not_a_string", str(raw)); continue

        # 1. normalise -------------------------------------------------
        text = normalize_text(raw, collapse_repeats=collapse_repeats)
        if text != raw:
            stats["docs_changed_by_normalisation"] += 1

        # 2. cheap length pre-check (so tiny docs are reported as "too short",
        #    not mislabelled as non-English)
        if text.count(" ") + 1 < min_words:
            drop("dropped_too_short", text); continue

        # 3. language --------------------------------------------------
        if language_filter:
            latin, en, script = language_profile(text)
            if latin < min_latin_ratio:
                dropped_scripts[script] += 1
                drop("dropped_non_latin_script", text); continue
            if en < min_english_ratio:
                dropped_scripts["latin-non-english"] += 1
                drop("dropped_non_english_latin", text); continue
            if strip_foreign_runs:
                cleaned = _FOREIGN_RUN_RE.sub(" ", text)
                if cleaned != text:
                    stats["docs_with_foreign_fragments_stripped"] += 1
                    text = _WS_RE.sub(" ", cleaned).strip()

        # 4. final length check (after foreign fragments were stripped) -
        words = _WORD_RE.findall(text.lower())
        if len(words) < min_words:
            drop("dropped_too_short", text); continue

        # 5. exact duplicates ------------------------------------------
        if exact_dedup:
            key = hashlib.blake2b(" ".join(words).encode(), digest_size=16).digest()
            if key in seen_exact:
                drop("dropped_exact_duplicate", text); continue
            seen_exact.add(key)

        # 6. near duplicates -------------------------------------------
        if near is not None and near.check_and_add(words):
            drop("dropped_near_duplicate", text); continue

        kept.append(text)
        if progress_every and n % progress_every == 0:
            print(f"  processed {n:,} | kept {len(kept):,}")

    stats["kept_docs"] = len(kept)
    return kept, {"stats": stats, "dropped_scripts": dropped_scripts, "examples": examples}


def print_clean_report(report, chars_before=None, chars_after=None):
    s = report["stats"]
    n = max(s["input_docs"], 1)
    print("=" * 62)
    print(f"Input documents                : {s['input_docs']:>9,}")
    for k in ["not_a_string", "dropped_non_latin_script", "dropped_non_english_latin",
              "dropped_too_short", "dropped_exact_duplicate", "dropped_near_duplicate"]:
        if s[k]:
            print(f"  - {k:<28}: {s[k]:>9,}  ({100 * s[k] / n:5.1f}%)")
    print(f"Kept documents                 : {s['kept_docs']:>9,}  ({100 * s['kept_docs'] / n:5.1f}%)")
    print(f"Docs changed by normalisation  : {s['docs_changed_by_normalisation']:>9,}")
    print(f"Docs with foreign text stripped: {s['docs_with_foreign_fragments_stripped']:>9,}")
    if chars_before and chars_after:
        print(f"Characters                     : {chars_before:,} -> {chars_after:,}")
    if report["dropped_scripts"]:
        print("\nLanguages / scripts removed:")
        for k, v in report["dropped_scripts"].most_common():
            print(f"  {k:<20}{v:>9,}")
    for reason, ex in report["examples"].items():
        print(f"\n[{reason}] sample:")
        for e in ex:
            print("   ", repr(e[:140]))
    print("=" * 62)


In [ ]:
chars_before = sum(len(t) for t in train_val_texts)

train_val_texts, clean_report = clean_corpus(
    train_val_texts,
    min_words=30,
    language_filter=True,
    min_latin_ratio=0.90,
    min_english_ratio=0.12,
    strip_foreign_runs=True,
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,
)

chars_after = sum(len(t) for t in train_val_texts)
print_clean_report(clean_report, chars_before, chars_after)

# free the raw per-source lists (they still hold the uncleaned text in RAM)
import gc
del acts, high_courts, supreme_court, tribunals
gc.collect()

rng = random.Random(42)
# rng.shuffle(train_val_texts)
print(f"Total documents after cleaning: {len(train_val_texts):,}")

  processed 20,000 | kept 19,035
  processed 40,000 | kept 38,546
  processed 60,000 | kept 57,767
  processed 80,000 | kept 76,848
  processed 100,000 | kept 95,793
  processed 120,000 | kept 114,672
  processed 140,000 | kept 133,464
  processed 160,000 | kept 152,248
  processed 180,000 | kept 170,886
  processed 200,000 | kept 189,472
  processed 220,000 | kept 208,028
  processed 240,000 | kept 226,656
  processed 260,000 | kept 245,267
  processed 280,000 | kept 263,838
  processed 300,000 | kept 282,247
  processed 320,000 | kept 300,726
  processed 340,000 | kept 320,447
  processed 360,000 | kept 340,120
  processed 380,000 | kept 359,020
  processed 400,000 | kept 377,322
  processed 420,000 | kept 395,368
  processed 460,000 | kept 430,799
  processed 500,000 | kept 466,238
  processed 520,000 | kept 483,782
  processed 540,000 | kept 501,796
  processed 560,000 | kept 519,796
  processed 600,000 | kept 555,583
  processed 620,000 | kept 573,403
  processed 640,000 | kept 59

In [ ]:
# ============================================================
# DATA CLEANING
#   1. Unicode + whitespace normalisation
#   2. Language / script filtering (keep English, drop the rest)
#   3. De-duplication (exact + near-duplicate via MinHash-LSH)
#
# Run this AFTER the raw documents are sampled (train_val_texts)
# and BEFORE the train / val split, so that duplicates cannot
# leak from train into validation.
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text


# ------------------------------------------------------------
# 2. LANGUAGE / SCRIPT HANDLING (no external dependency)
# ------------------------------------------------------------
_LATIN = "A-Za-z\u00C0-\u024F\u1E00-\u1EFF"
_SCRIPTS = {
    "devanagari": "\u0900-\u097F", "bengali": "\u0980-\u09FF",
    "gurmukhi":   "\u0A00-\u0A7F", "gujarati": "\u0A80-\u0AFF",
    "odia":       "\u0B00-\u0B7F", "tamil":    "\u0B80-\u0BFF",
    "telugu":     "\u0C00-\u0C7F", "kannada":  "\u0C80-\u0CFF",
    "malayalam":  "\u0D00-\u0D7F",
    "arabic/urdu": "\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF",
    "cyrillic":   "\u0400-\u04FF",
    "cjk":        "\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF",
}
_SCRIPT_RES = {k: re.compile(f"[{v}]") for k, v in _SCRIPTS.items()}
_ALPHA_RE = re.compile(r"[^\W\d_]")
_LATIN_RE = re.compile(f"[{_LATIN}]")
# a run of non-Latin letters (plus the combining marks / joiners that belong to them)
_FOREIGN_RUN_RE = re.compile(
    "[" + "".join(_SCRIPTS.values()) + "]+"
    "(?:[\\s\u200c\u200d]*[" + "".join(_SCRIPTS.values()) + "]+)*"
)
_EN_WORD_RE = re.compile(r"[a-z]+")
_ENGLISH_STOP = frozenset("""
the of and to in that is was for by on with as be it at this which are from or not has have
had been an a he she they his her their its were any such under also but if all no can shall
may would
""".split())


def language_profile(text, sample_chars=3000):
    """Cheap profile from the start and middle of a document.
    Returns (latin_ratio, english_stopword_ratio, dominant_foreign_script)."""
    n = len(text)
    s = text if n <= 2 * sample_chars else (
        text[:sample_chars] + " " + text[n // 2: n // 2 + sample_chars])
    n_alpha = len(_ALPHA_RE.findall(s))
    if n_alpha == 0:
        return 0.0, 0.0, "none"
    n_latin = len(_LATIN_RE.findall(s))
    latin_ratio = n_latin / n_alpha

    words = _EN_WORD_RE.findall(s.lower())
    en_ratio = (sum(w in _ENGLISH_STOP for w in words) / len(words)) if words else 0.0

    foreign = "latin-non-english"
    if latin_ratio < 0.5:
        counts = {k: len(r.findall(s)) for k, r in _SCRIPT_RES.items()}
        best = max(counts, key=counts.get)
        foreign = best if counts[best] > 0 else "other"
    return latin_ratio, en_ratio, foreign


# ------------------------------------------------------------
# 3. NEAR-DUPLICATE DETECTION (MinHash + LSH banding)
# ------------------------------------------------------------
_WORD_RE = re.compile(r"\w+")
_MH_PRIME = np.uint64(4294967311)           # prime just above 2**32
_ROLL = np.uint64(1000003)


class NearDupIndex:
    """Streaming near-duplicate detector. The first copy seen is kept."""

    def __init__(self, num_perm=64, bands=8, threshold=0.8,
                 shingle=5, max_words=2000, seed=1):
        assert num_perm % bands == 0
        self.num_perm, self.bands = num_perm, bands
        self.rows = num_perm // bands
        self.threshold, self.shingle, self.max_words = threshold, shingle, max_words
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.b = rng.randint(0, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.sigs = []
        self.buckets = [dict() for _ in range(bands)]

    def _shingle_hashes(self, words):
        words = words[: self.max_words]
        k = self.shingle
        wh = np.fromiter((zlib.crc32(w.encode()) for w in words),
                         dtype=np.uint64, count=len(words))
        m = len(wh) - k + 1
        if m < 1:
            return wh
        sh = wh[:m].copy()
        for j in range(1, k):                 # rolling hash of k consecutive words
            sh = sh * _ROLL + wh[j: m + j]
        return (sh ^ (sh >> np.uint64(32))) & np.uint64(0xFFFFFFFF)

    def check_and_add(self, words):
        """True if a near-duplicate was already indexed; otherwise index and return False."""
        h = self._shingle_hashes(words)[None, :]
        sig = ((self.a * h + self.b) % _MH_PRIME).min(axis=1)
        r = self.rows
        keys = [hash(sig[i * r:(i + 1) * r].tobytes()) for i in range(self.bands)]
        for i, k in enumerate(keys):
            j = self.buckets[i].get(k)
            if j is not None and (self.sigs[j] == sig).mean() >= self.threshold:
                return True
        idx = len(self.sigs)
        self.sigs.append(sig)
        for i, k in enumerate(keys):
            self.buckets[i].setdefault(k, idx)
        return False


# ------------------------------------------------------------
# PIPELINE
# ------------------------------------------------------------
def clean_corpus(
    texts,
    min_words=30,                 # drop near-empty documents
    language_filter=True,         # drop non-English documents
    min_latin_ratio=0.90,         # share of letters that must be Latin script
    min_english_ratio=0.12,       # share of words that are common English words
    strip_foreign_runs=True,      # remove stray Hindi/Urdu/... fragments inside English docs
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,          # estimated Jaccard similarity above which = duplicate
    collapse_repeats=True,
    n_examples=3,
    progress_every=20_000,
):
    stats = collections.Counter()
    dropped_scripts = collections.Counter()
    examples = collections.defaultdict(list)
    seen_exact = set()
    near = NearDupIndex(threshold=near_threshold) if near_dedup else None
    kept = []

    def drop(reason, text):
        stats[reason] += 1
        if len(examples[reason]) < n_examples:
            examples[reason].append(text[:200])

    for n, raw in enumerate(texts, 1):
        stats["input_docs"] += 1
        if not isinstance(raw, str):
            drop("not_a_string", str(raw)); continue

        # 1. normalise -------------------------------------------------
        text = normalize_text(raw, collapse_repeats=collapse_repeats)
        if text != raw:
            stats["docs_changed_by_normalisation"] += 1

        # 2. cheap length pre-check (so tiny docs are reported as "too short",
        #    not mislabelled as non-English)
        if text.count(" ") + 1 < min_words:
            drop("dropped_too_short", text); continue

        # 3. language --------------------------------------------------
        if language_filter:
            latin, en, script = language_profile(text)
            if latin < min_latin_ratio:
                dropped_scripts[script] += 1
                drop("dropped_non_latin_script", text); continue
            if en < min_english_ratio:
                dropped_scripts["latin-non-english"] += 1
                drop("dropped_non_english_latin", text); continue
            if strip_foreign_runs:
                cleaned = _FOREIGN_RUN_RE.sub(" ", text)
                if cleaned != text:
                    stats["docs_with_foreign_fragments_stripped"] += 1
                    text = _WS_RE.sub(" ", cleaned).strip()

        # 4. final length check (after foreign fragments were stripped) -
        words = _WORD_RE.findall(text.lower())
        if len(words) < min_words:
            drop("dropped_too_short", text); continue

        # 5. exact duplicates ------------------------------------------
        if exact_dedup:
            key = hashlib.blake2b(" ".join(words).encode(), digest_size=16).digest()
            if key in seen_exact:
                drop("dropped_exact_duplicate", text); continue
            seen_exact.add(key)

        # 6. near duplicates -------------------------------------------
        if near is not None and near.check_and_add(words):
            drop("dropped_near_duplicate", text); continue

        kept.append(text)
        if progress_every and n % progress_every == 0:
            print(f"  processed {n:,} | kept {len(kept):,}")

    stats["kept_docs"] = len(kept)
    return kept, {"stats": stats, "dropped_scripts": dropped_scripts, "examples": examples}


def print_clean_report(report, chars_before=None, chars_after=None):
    s = report["stats"]
    n = max(s["input_docs"], 1)
    print("=" * 62)
    print(f"Input documents                : {s['input_docs']:>9,}")
    for k in ["not_a_string", "dropped_non_latin_script", "dropped_non_english_latin",
              "dropped_too_short", "dropped_exact_duplicate", "dropped_near_duplicate"]:
        if s[k]:
            print(f"  - {k:<28}: {s[k]:>9,}  ({100 * s[k] / n:5.1f}%)")
    print(f"Kept documents                 : {s['kept_docs']:>9,}  ({100 * s['kept_docs'] / n:5.1f}%)")
    print(f"Docs changed by normalisation  : {s['docs_changed_by_normalisation']:>9,}")
    print(f"Docs with foreign text stripped: {s['docs_with_foreign_fragments_stripped']:>9,}")
    if chars_before and chars_after:
        print(f"Characters                     : {chars_before:,} -> {chars_after:,}")
    if report["dropped_scripts"]:
        print("\nLanguages / scripts removed:")
        for k, v in report["dropped_scripts"].most_common():
            print(f"  {k:<20}{v:>9,}")
    for reason, ex in report["examples"].items():
        print(f"\n[{reason}] sample:")
        for e in ex:
            print("   ", repr(e[:140]))
    print("=" * 62)


In [ ]:
# Determine split index based on VAL_FRAC
val_size = int(len(train_val_texts) * VAL_FRAC)
train_size = len(train_val_texts) - val_size

# Wrap documents in dictionary structures with 'text' key to match downstream expectations
train_docs = [{"text": doc} for doc in train_val_texts[:train_size]]
val_docs = [{"text": doc} for doc in train_val_texts[train_size:]]

print(f"Total documents: {len(train_val_texts):,}")
print(f"Train documents (train_docs): {len(train_docs):,} ({100 * len(train_docs)/len(train_val_texts):.1f}%)")
print(f"Validation documents (val_docs): {len(val_docs):,} ({100 * len(val_docs)/len(train_val_texts):.1f}%)")


Total documents: 613,517
Train documents (train_docs): 582,842 (95.0%)
Validation documents (val_docs): 30,675 (5.0%)


In [ ]:
# Determine split index based on VAL_FRAC
val_size = int(len(train_val_texts) * VAL_FRAC)
train_size = len(train_val_texts) - val_size

# Wrap documents in dictionary structures with 'text' key to match downstream expectations
train_docs = [{"text": doc} for doc in train_val_texts[:train_size]]
val_docs = [{"text": doc} for doc in train_val_texts[train_size:]]

print(f"Total documents: {len(train_val_texts):,}")
print(f"Train documents (train_docs): {len(train_docs):,} ({100 * len(train_docs)/len(train_val_texts):.1f}%)")
print(f"Validation documents (val_docs): {len(val_docs):,} ({100 * len(val_docs)/len(train_val_texts):.1f}%)")


Total documents: 670
Train documents (train_docs): 637 (95.1%)
Validation documents (val_docs): 33 (4.9%)


In [ ]:
vocab_size = 32768
tokenizer = BPETokenizer(vocab_size=vocab_size)

print(f"Training BPE tokenizer with vocab size {vocab_size}...")

tokenizer = BPETokenizer(
    vocab_size=vocab_size,
    reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
)
tokenizer.train(
    train_val_texts,
    save_checkpoints=[8192, 16384, 32768],
    checkpoint_dir=WORK_DIR
)

print("Tokenizer training finished!")

bpe_en_path = os.path.join(WORK_DIR, 'final_bpe_en.json')
tokenizer.save(bpe_en_path)


Training BPE tokenizer with vocab size 32768...

STEP 1: word frequencies
  [warn] example 693 has 627,479 chars -> truncated to 500,000
Examples: 1,000 | Unique words: 38,852 | RAM: 12.26 GB
  [warn] example 1,435 has 778,765 chars -> truncated to 500,000
Examples: 2,000 | Unique words: 56,855 | RAM: 12.26 GB
Examples: 3,000 | Unique words: 72,440 | RAM: 12.26 GB
  [warn] example 3,865 has 1,210,030 chars -> truncated to 500,000
Examples: 4,000 | Unique words: 90,245 | RAM: 12.26 GB
Examples: 5,000 | Unique words: 54,218 | RAM: 12.26 GB
  [warn] example 5,851 has 566,843 chars -> truncated to 500,000
Examples: 6,000 | Unique words: 70,548 | RAM: 12.26 GB
Examples: 7,000 | Unique words: 85,318 | RAM: 12.26 GB
Examples: 8,000 | Unique words: 94,856 | RAM: 12.26 GB
Examples: 9,000 | Unique words: 104,938 | RAM: 12.26 GB
Examples: 10,000 | Unique words: 75,550 | RAM: 12.26 GB
  [warn] example 10,902 has 615,629 chars -> truncated to 500,000
Examples: 11,000 | Unique words: 88,397 | RAM: 1

In [ ]:
# Load the trained tokenizer from the completed file
tokenizer = BPETokenizer.from_json(os.path.join(WORK_DIR,'final_bpe_en.json'))

# Configure global vocabulary ID mappings
PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
vocab_size = len(tokenizer.itos)

print("Tokenizer loaded successfully!")
print(f"Vocabulary size: {vocab_size:,}")
print(f"ID mappings -> PAD: {PAD_ID}, UNK: {UNK_ID}, BOS: {BOS_ID}, EOS: {EOS_ID}")

Tokenizer loaded successfully!
Vocabulary size: 32,768
ID mappings -> PAD: 0, UNK: 1, BOS: 2, EOS: 3


## 3. Document-level train/validation split

In [ ]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step


## 4. Custom project BPE tokenizer

The class below is copied from the Transformer-XL notebook's custom `BPETokenizer` implementation. This ensures the same tokenization algorithm and JSON format are used across experiments. We train on **train documents only** to avoid validation leakage.

## 5. Build prompt → continuation examples using BPE lengths

In [ ]:
# Word-level windows create readable prompt/continuation boundaries; then each side
# is encoded and clipped by BPE token length. Documents never cross split boundaries.
PROMPT_WORDS = 350
CONTINUATION_WORDS = 100
STRIDE_WORDS = 175

def make_windows(text, prompt_words=PROMPT_WORDS,
                 continuation_words=CONTINUATION_WORDS, stride=STRIDE_WORDS):
    words = text.split()
    total = prompt_words + continuation_words
    examples = []
    if len(words) < total:
        # Retain shorter legal documents by using a proportional split.
        if len(words) >= 80:
            split = max(40, int(0.75 * len(words)))
            examples.append((" ".join(words[:split]), " ".join(words[split:])))
        return examples
    for start in range(0, len(words) - total + 1, stride):
        chunk = words[start:start + total]
        examples.append((" ".join(chunk[:prompt_words]), " ".join(chunk[prompt_words:])))
    return examples

def build_examples(docs):
    out = []
    for doc in docs:
        out.extend(make_windows(doc["text"]))
    return out

train_examples = build_examples(train_docs)
val_examples = build_examples(val_docs)
print(f"Train examples: {len(train_examples):,} | val examples: {len(val_examples):,}")
if not train_examples or not val_examples:
    raise ValueError("No prompt/continuation examples were created; inspect corpus and window sizes.")

Train examples: 3,314,527 | val examples: 116,540


In [ ]:
# Word-level windows create readable prompt/continuation boundaries; then each side
# is encoded and clipped by BPE token length. Documents never cross split boundaries.
PROMPT_WORDS = 350
CONTINUATION_WORDS = 100
STRIDE_WORDS = 175

def make_windows(text, prompt_words=PROMPT_WORDS,
                 continuation_words=CONTINUATION_WORDS, stride=STRIDE_WORDS):
    words = text.split()
    total = prompt_words + continuation_words
    examples = []
    if len(words) < total:
        # Retain shorter legal documents by using a proportional split.
        if len(words) >= 80:
            split = max(40, int(0.75 * len(words)))
            examples.append((" ".join(words[:split]), " ".join(words[split:])))
        return examples
    for start in range(0, len(words) - total + 1, stride):
        chunk = words[start:start + total]
        examples.append((" ".join(chunk[:prompt_words]), " ".join(chunk[prompt_words:])))
    return examples

def build_examples(docs):
    out = []
    for doc in docs:
        out.extend(make_windows(doc["text"]))
    return out

train_examples = build_examples(train_docs)
val_examples = build_examples(val_docs)
print(f"Train examples: {len(train_examples):,} | val examples: {len(val_examples):,}")
if not train_examples or not val_examples:
    raise ValueError("No prompt/continuation examples were created; inspect corpus and window sizes.")

Train examples: 3,728 | val examples: 216


In [ ]:
class Seq2SeqPretrainDataset(Dataset):
    def __init__(self, examples, tokenizer, max_enc_len=MAX_ENC_LEN, max_dec_len=MAX_DEC_LEN):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_enc_len = max_enc_len
        self.max_dec_len = max_dec_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        prompt, continuation = self.examples[idx]
        # Add EOS to encoder input and target. BOS is added during collate.
        enc_ids = self.tokenizer.encode(prompt, add_special_tokens=False)[:self.max_enc_len - 1]
        enc_ids = enc_ids + [EOS_ID]
        target_ids = self.tokenizer.encode(continuation, add_special_tokens=False)[:self.max_dec_len - 1]
        target_ids = target_ids + [EOS_ID]
        return {"enc_ids": enc_ids, "target_ids": target_ids}

def collate_seq2seq(batch):
    batch_size = len(batch)
    max_enc = max(len(x["enc_ids"]) for x in batch)
    max_target = max(len(x["target_ids"]) for x in batch)

    input_ids = torch.full((batch_size, max_enc), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_enc), dtype=torch.long)
    decoder_input_ids = torch.full((batch_size, max_target), PAD_ID, dtype=torch.long)
    decoder_attention_mask = torch.zeros((batch_size, max_target), dtype=torch.long)
    labels = torch.full((batch_size, max_target), -100, dtype=torch.long)

    for i, item in enumerate(batch):
        enc = item["enc_ids"]
        target = item["target_ids"]
        # Correct teacher-forcing shift: decoder input is BOS + target[:-1],
        # labels are target. Thus the first decoder position predicts target[0].
        dec_in = [BOS_ID] + target[:-1]
        input_ids[i, :len(enc)] = torch.tensor(enc)
        attention_mask[i, :len(enc)] = 1
        decoder_input_ids[i, :len(dec_in)] = torch.tensor(dec_in)
        decoder_attention_mask[i, :len(dec_in)] = 1
        labels[i, :len(target)] = torch.tensor(target)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_attention_mask,
        "labels": labels,
    }

train_loader = DataLoader(
    Seq2SeqPretrainDataset(train_examples, tokenizer),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_seq2seq,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    Seq2SeqPretrainDataset(val_examples, tokenizer),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_seq2seq,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Non-ignored target tokens:", int((batch["labels"] != -100).sum()))

{'input_ids': (2, 411), 'attention_mask': (2, 411), 'decoder_input_ids': (2, 115), 'decoder_attention_mask': (2, 115), 'labels': (2, 115)}
Non-ignored target tokens: 229


In [ ]:
class Seq2SeqPretrainDataset(Dataset):
    def __init__(self, examples, tokenizer, max_enc_len=MAX_ENC_LEN, max_dec_len=MAX_DEC_LEN):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_enc_len = max_enc_len
        self.max_dec_len = max_dec_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        prompt, continuation = self.examples[idx]
        # Add EOS to encoder input and target. BOS is added during collate.
        enc_ids = self.tokenizer.encode(prompt, add_special_tokens=False)[:self.max_enc_len - 1]
        enc_ids = enc_ids + [EOS_ID]
        target_ids = self.tokenizer.encode(continuation, add_special_tokens=False)[:self.max_dec_len - 1]
        target_ids = target_ids + [EOS_ID]
        return {"enc_ids": enc_ids, "target_ids": target_ids}

def collate_seq2seq(batch):
    batch_size = len(batch)
    max_enc = max(len(x["enc_ids"]) for x in batch)
    max_target = max(len(x["target_ids"]) for x in batch)

    input_ids = torch.full((batch_size, max_enc), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_enc), dtype=torch.long)
    decoder_input_ids = torch.full((batch_size, max_target), PAD_ID, dtype=torch.long)
    decoder_attention_mask = torch.zeros((batch_size, max_target), dtype=torch.long)
    labels = torch.full((batch_size, max_target), -100, dtype=torch.long)

    for i, item in enumerate(batch):
        enc = item["enc_ids"]
        target = item["target_ids"]
        # Correct teacher-forcing shift: decoder input is BOS + target[:-1],
        # labels are target. Thus the first decoder position predicts target[0].
        dec_in = [BOS_ID] + target[:-1]
        input_ids[i, :len(enc)] = torch.tensor(enc)
        attention_mask[i, :len(enc)] = 1
        decoder_input_ids[i, :len(dec_in)] = torch.tensor(dec_in)
        decoder_attention_mask[i, :len(dec_in)] = 1
        labels[i, :len(target)] = torch.tensor(target)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_attention_mask,
        "labels": labels,
    }

train_loader = DataLoader(
    Seq2SeqPretrainDataset(train_examples, tokenizer),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_seq2seq,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    Seq2SeqPretrainDataset(val_examples, tokenizer),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_seq2seq,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Non-ignored target tokens:", int((batch["labels"] != -100).sum()))

{'input_ids': (2, 412), 'attention_mask': (2, 412), 'decoder_input_ids': (2, 124), 'decoder_attention_mask': (2, 124), 'labels': (2, 124)}
Non-ignored target tokens: 236


## 6. Encoder–Decoder model (RMSNorm, relative position bias, gated FFN)

In [ ]:
D_MODEL = 512
NUM_HEADS = 16
D_FF = 2048
NUM_ENCODER_LAYERS = 9
NUM_DECODER_LAYERS = 9
DROPOUT = 0.1


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        var = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return self.weight * x


class RelativePositionBias(nn.Module):
    """T5-style relative position bias, bucketed on a log scale for far-apart positions."""

    def __init__(self, num_heads, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def _bucket(relative_position, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -relative_position
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).to(torch.long) * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_if_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6)
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        val_if_large = torch.clamp(val_if_large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, val_if_large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        bucket = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(bucket)              # q_len, k_len, num_heads
        return values.permute(2, 0, 1).unsqueeze(0)    # 1, num_heads, q_len, k_len


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, mask=None, position_bias=None):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q(x_q).view(B, Lq, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if position_bias is not None:
            scores = scores + position_bias
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Lq, D)
        return self.o(out)


class GatedFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.wi_0 = nn.Linear(d_model, d_ff, bias=False)
        self.wi_1 = nn.Linear(d_model, d_ff, bias=False)
        self.wo = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.wi_0(x)) * self.wi_1(x)
        return self.wo(self.dropout(h))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, position_bias):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, mask, position_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.ff(h))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln3 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask, self_pos_bias):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, self_mask, self_pos_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, cross_mask, None))
        h = self.ln3(x)
        x = x + self.dropout(self.ff(h))
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=True)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask):
        L = x.shape[1]
        pos_bias = self.pos_bias(L, L, x.device)
        mask = attention_mask[:, None, None, :]
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, mask, pos_bias)
        return self.final_ln(x)


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=False)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_attention_mask, cross_attention_mask):
        L = x.shape[1]
        causal = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool))
        pad = self_attention_mask[:, None, None, :].bool()
        self_mask = (causal[None, None] & pad).float()
        cross_mask = cross_attention_mask[:, None, None, :]
        pos_bias = self.pos_bias(L, L, x.device)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, enc_out, self_mask, cross_mask, pos_bias)
        return self.final_ln(x)


class LegalSeq2SeqLM(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF,
                 num_encoder_layers=NUM_ENCODER_LAYERS, num_decoder_layers=NUM_DECODER_LAYERS,
                 dropout=DROPOUT, pad_id=PAD_ID):
        super().__init__()
        self.shared_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_encoder_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_decoder_layers, dropout)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.shared_embed.weight  # tied embeddings

    def forward(self, input_ids, attention_mask, decoder_input_ids, decoder_attention_mask):
        enc_out = self.encoder(self.shared_embed(input_ids), attention_mask)
        dec_out = self.decoder(
            self.shared_embed(decoder_input_ids), enc_out, decoder_attention_mask, attention_mask
        )
        return self.lm_head(dec_out)

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = LegalSeq2SeqLM(VOCAB_SIZE).to(DEVICE)
n_params = model.num_parameters()
print(f"Model parameters: {n_params:,}  ({n_params/1e6:.1f}M)")
# If this isn't close enough to 100M for your purposes, adjust D_MODEL / D_FF /
# NUM_ENCODER_LAYERS / NUM_DECODER_LAYERS above and re-run this cell.

Model parameters: 101,736,960  (101.7M)


## 7. Training, token-weighted validation, checkpoint resume

In [ ]:
def move_batch(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}

def make_amp_context(device, enabled=True):
    return torch.autocast(
        device_type=device.type,
        dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
        enabled=bool(enabled and device.type in ("cuda", "cpu") and (device.type == "cuda")),
    )

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, global_step,
                    best_val_loss, history):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "scaler_state_dict": scaler.state_dict() if scaler.is_enabled() else None,
        "epoch": epoch, "global_step": global_step,
        "best_val_loss": best_val_loss, "history": history,
        "model_config": {
            "vocab_size": vocab_size, "d_model": D_MODEL, "num_heads": NUM_HEADS,
            "d_ff": D_FF, "num_encoder_layers": NUM_ENCODER_LAYERS,
            "num_decoder_layers": NUM_DECODER_LAYERS, "dropout": DROPOUT,
            "pad_id": PAD_ID,
        },
        "tokenizer_path": TOKENIZER_PATH,
    }, path)

@torch.no_grad()
def validate_seq2seq(model, loader, max_batches=None):
    was_training = model.training
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for idx, batch in enumerate(loader):
        if max_batches is not None and idx >= max_batches:
            break
        batch = move_batch(batch, DEVICE)
        with make_amp_context(DEVICE, USE_AMP):
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch["decoder_input_ids"], batch["decoder_attention_mask"])
        labels = batch["labels"]
        nll = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                              labels.reshape(-1), ignore_index=-100, reduction="sum")
        total_nll += float(nll.item())
        total_tokens += int((labels != -100).sum().item())
    if was_training:
        model.train()
    return total_nll / max(total_tokens, 1)

model = LegalSeq2SeqLM(vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,} ({n_params / 1e6:.1f}M)")

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98),
                  eps=1e-8, weight_decay=WEIGHT_DECAY)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = max(1, updates_per_epoch * EPOCHS)

def lr_lambda(step):
    if step < max(1, WARMUP_STEPS):
        return max(1, step) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))
LATEST_CKPT = os.path.join(CHECKPOINT_DIR, "encoder_decoder_latest.pt")
BEST_CKPT = os.path.join(CHECKPOINT_DIR, "encoder_decoder_best.pt")

start_epoch, global_step, best_val_loss, history = 0, 0, float("inf"), []
if os.path.exists(LATEST_CKPT):
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and ckpt.get("scheduler_state_dict"):
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler.is_enabled() and ckpt.get("scaler_state_dict"):
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = int(ckpt.get("epoch", 0))
    global_step = int(ckpt.get("global_step", 0))
    best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
    history = ckpt.get("history", [])
    print(f"Resuming at epoch={start_epoch}, optimizer step={global_step}")
else:
    print("No checkpoint found; starting from scratch.")

Trainable parameters: 101,736,960 (101.7M)
Resuming at epoch=0, optimizer step=11200


In [ ]:
def train_seq2seq(model, train_loader, val_loader):
    global global_step, best_val_loss, history, start_epoch
    optimizer.zero_grad(set_to_none=True)
    running_loss, running_batches = 0.0, 0
    t0 = time.time()

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss_sum, epoch_tokens = 0.0, 0
        for batch_idx, batch in enumerate(train_loader):
            if MAX_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_BATCHES_PER_EPOCH:
                print(f"Reached MAX_BATCHES_PER_EPOCH={MAX_BATCHES_PER_EPOCH}")
                break

            batch = move_batch(batch, DEVICE)
            with make_amp_context(DEVICE, USE_AMP):
                logits = model(batch["input_ids"], batch["attention_mask"],
                               batch["decoder_input_ids"], batch["decoder_attention_mask"])
                labels = batch["labels"]
                loss_sum = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                           labels.reshape(-1), ignore_index=-100, reduction="sum")
                n_tokens = (labels != -100).sum().clamp_min(1)
                loss = loss_sum / n_tokens

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                raise FloatingPointError(
                    f"Non-finite loss at epoch={epoch}, batch={batch_idx}. "
                    "Inspect tokenizer IDs, masks, and learning rate."
                )

            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            running_loss += float(loss.detach().item())
            running_batches += 1
            epoch_loss_sum += float(loss_sum.detach().item())
            epoch_tokens += int(n_tokens.item())

            do_update = ((batch_idx + 1) % GRAD_ACCUM_STEPS == 0) or (batch_idx + 1 == len(train_loader))
            if do_update:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if not torch.isfinite(grad_norm):
                    optimizer.zero_grad(set_to_none=True)
                    raise FloatingPointError("Non-finite gradient norm; reduce LR or inspect model.")
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

                if global_step % LOG_EVERY == 0:
                    avg = running_loss / max(running_batches, 1)
                    print(f"epoch={epoch+1} step={global_step} loss={avg:.4f} "
                          f"ppl={math.exp(min(avg, 20)):.2f} grad_norm={float(grad_norm):.3f} "
                          f"lr={scheduler.get_last_lr()[0]:.2e} elapsed={time.time()-t0:.0f}s")
                    running_loss, running_batches = 0.0, 0

                if global_step % VAL_EVERY == 0:
                    val_loss = validate_seq2seq(model, val_loader)
                    val_ppl = math.exp(min(val_loss, 20))
                    history.append({"step": global_step, "epoch": epoch, "val_loss": val_loss, "val_ppl": val_ppl})
                    print(f"[validation] step={global_step} token_loss={val_loss:.4f} ppl={val_ppl:.2f}")
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, scaler,
                                        epoch, global_step, best_val_loss, history)
                        print("Saved best checkpoint:", BEST_CKPT)

                if global_step % SAVE_EVERY == 0:
                    save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, scaler,
                                    epoch, global_step, best_val_loss, history)
                    print("Saved resumable checkpoint:", LATEST_CKPT)

        train_loss = epoch_loss_sum / max(epoch_tokens, 1)
        val_loss = validate_seq2seq(model, val_loader)
        val_ppl = math.exp(min(val_loss, 20))
        history.append({"step": global_step, "epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_ppl": val_ppl})
        print(f"[epoch {epoch+1}] train_token_loss={train_loss:.4f} "
              f"val_token_loss={val_loss:.4f} val_ppl={val_ppl:.2f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CKPT, model, optimizer, scheduler, scaler,
                            epoch + 1, global_step, best_val_loss, history)
            print("Saved best checkpoint:", BEST_CKPT)
        save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, scaler,
                        epoch + 1, global_step, best_val_loss, history)

    start_epoch = EPOCHS
    return history

history = train_seq2seq(model, train_loader, val_loader)

In [ ]:
def train_seq2seq(model, train_loader, val_loader):
    global global_step, best_val_loss, history, start_epoch
    optimizer.zero_grad(set_to_none=True)
    running_loss, running_batches = 0.0, 0
    t0 = time.time()

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss_sum, epoch_tokens = 0.0, 0
        for batch_idx, batch in enumerate(train_loader):
            if MAX_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_BATCHES_PER_EPOCH:
                print(f"Reached MAX_BATCHES_PER_EPOCH={MAX_BATCHES_PER_EPOCH}")
                break

            batch = move_batch(batch, DEVICE)
            with make_amp_context(DEVICE, USE_AMP):
                logits = model(batch["input_ids"], batch["attention_mask"],
                               batch["decoder_input_ids"], batch["decoder_attention_mask"])
                labels = batch["labels"]
                loss_sum = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                           labels.reshape(-1), ignore_index=-100, reduction="sum")
                n_tokens = (labels != -100).sum().clamp_min(1)
                loss = loss_sum / n_tokens

            if not torch.isfinite(loss):
                print(f"[warning] Non-finite loss at epoch={epoch}, batch={batch_idx}. Skipping batch.")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            running_loss += float(loss.detach().item())
            running_batches += 1
            epoch_loss_sum += float(loss_sum.detach().item())
            epoch_tokens += int(n_tokens.item())

            do_update = ((batch_idx + 1) % GRAD_ACCUM_STEPS == 0) or (batch_idx + 1 == len(train_loader))
            if do_update:
                # Only unscale if we haven't already unscaled this step
                try:
                    scaler.unscale_(optimizer)
                except RuntimeError as e:
                    # Catch the error if already unscaled in this step
                    if "unscale_() has already been called" not in str(e):
                        raise e

                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if not torch.isfinite(grad_norm):
                    print(f"[warning] Non-finite gradient norm ({float(grad_norm)}) at step {global_step}. Skipping step and scaling down.")
                    optimizer.zero_grad(set_to_none=True)
                    scaler.update() # This will scale down the loss scaling factor
                    continue

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

                if global_step % LOG_EVERY == 0:
                    avg = running_loss / max(running_batches, 1)
                    print(f"epoch={epoch+1} step={global_step} loss={avg:.4f} "
                          f"ppl={math.exp(min(avg, 20)):.2f} grad_norm={float(grad_norm):.3f} "
                          f"lr={scheduler.get_last_lr()[0]:.2e} elapsed={time.time()-t0:.0f}s")
                    running_loss, running_batches = 0.0, 0

                if global_step % VAL_EVERY == 0:
                    val_loss = validate_seq2seq(model, val_loader)
                    val_ppl = math.exp(min(val_loss, 20))
                    history.append({"step": global_step, "epoch": epoch, "val_loss": val_loss, "val_ppl": val_ppl})
                    print(f"[validation] step={global_step} token_loss={val_loss:.4f} ppl={val_ppl:.2f}")
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, scaler,
                                        epoch, global_step, best_val_loss, history)
                        print("Saved best checkpoint:", BEST_CKPT)

                if global_step % SAVE_EVERY == 0:
                    save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, scaler,
                                    epoch, global_step, best_val_loss, history)
                    print("Saved resumable checkpoint:", LATEST_CKPT)

        train_loss = epoch_loss_sum / max(epoch_tokens, 1)
        val_loss = validate_seq2seq(model, val_loader)
        val_ppl = math.exp(min(val_loss, 20))
        history.append({"step": global_step, "epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "val_ppl": val_ppl})
        print(f"[epoch {epoch+1}] train_token_loss={train_loss:.4f} "
              f"val_token_loss={val_loss:.4f} val_ppl={val_ppl:.2f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CKPT, model, optimizer, scheduler, scaler,
                            epoch + 1, global_step, best_val_loss, history)
            print("Saved best checkpoint:", BEST_CKPT)
        save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, scaler,
                        epoch + 1, global_step, best_val_loss, history)

    start_epoch = EPOCHS
    return history

In [ ]:
history = train_seq2seq(model, train_loader, val_loader)

epoch=1 step=3025 loss=6.9002 ppl=992.51 grad_norm=17.155 lr=1.00e-04 elapsed=32s
epoch=1 step=3050 loss=6.8218 ppl=917.64 grad_norm=12.185 lr=1.00e-04 elapsed=63s
epoch=1 step=3075 loss=6.8226 ppl=918.34 grad_norm=14.485 lr=1.00e-04 elapsed=93s
epoch=1 step=3100 loss=6.8516 ppl=945.37 grad_norm=13.167 lr=1.00e-04 elapsed=124s
epoch=1 step=3125 loss=6.7470 ppl=851.48 grad_norm=13.370 lr=1.00e-04 elapsed=154s
epoch=1 step=3150 loss=6.7275 ppl=835.06 grad_norm=12.480 lr=1.00e-04 elapsed=185s
epoch=1 step=3175 loss=6.6889 ppl=803.46 grad_norm=7.236 lr=1.00e-04 elapsed=215s
epoch=1 step=3200 loss=6.7296 ppl=836.79 grad_norm=16.335 lr=1.00e-04 elapsed=246s
[validation] step=3200 token_loss=6.3506 ppl=572.83
Saved best checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_best.pt
Saved resumable checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_latest.pt
epoch=1 step=3225 loss=6.6768 ppl

In [ ]:
history = train_seq2seq(model, train_loader, val_loader)

epoch=1 step=8025 loss=5.4930 ppl=242.98 grad_norm=9.502 lr=1.00e-04 elapsed=29s
epoch=1 step=8050 loss=5.5360 ppl=253.65 grad_norm=10.742 lr=1.00e-04 elapsed=59s
epoch=1 step=8075 loss=5.4879 ppl=241.76 grad_norm=9.020 lr=1.00e-04 elapsed=89s
epoch=1 step=8100 loss=5.3514 ppl=210.90 grad_norm=11.116 lr=1.00e-04 elapsed=119s
epoch=1 step=8125 loss=5.4213 ppl=226.16 grad_norm=8.913 lr=1.00e-04 elapsed=148s
epoch=1 step=8150 loss=5.3798 ppl=216.98 grad_norm=11.041 lr=1.00e-04 elapsed=178s
epoch=1 step=8175 loss=5.4138 ppl=224.49 grad_norm=6.792 lr=1.00e-04 elapsed=208s
epoch=1 step=8200 loss=5.4490 ppl=232.52 grad_norm=8.130 lr=1.00e-04 elapsed=238s
[validation] step=8200 token_loss=5.1402 ppl=170.74
Saved best checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_best.pt
Saved resumable checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_latest.pt
epoch=1 step=8225 loss=5.3778 ppl=216

## 8. Plot training/validation history

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)
display(history_df.tail(10))
if not history_df.empty and "val_loss" in history_df:
    plt.figure(figsize=(8, 4))
    plt.plot(history_df["step"], history_df["val_loss"], marker="o")
    plt.xlabel("Optimizer step"); plt.ylabel("Validation token loss")
    plt.title("Encoder–Decoder validation loss"); plt.grid(True); plt.show()

## 9. Load best checkpoint and generate continuations

In [ ]:
@torch.no_grad()
def generate_continuation(model, tokenizer, prompt_text, max_new_tokens=128,
                          temperature=1.0, top_k=50, top_p=0.9):
    model.eval()
    enc_ids = tokenizer.encode(prompt_text, add_special_tokens=False)[:MAX_ENC_LEN - 1] + [EOS_ID]
    input_ids = torch.tensor([enc_ids], dtype=torch.long, device=DEVICE)
    attention_mask = torch.ones_like(input_ids)
    enc_out = model.encoder(model.shared_embed(input_ids), attention_mask)
    decoder_ids = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)

    for _ in range(max_new_tokens):
        decoder_mask = torch.ones_like(decoder_ids)
        dec_out = model.decoder(model.shared_embed(decoder_ids), enc_out,
                                decoder_mask, attention_mask)
        logits = model.lm_head(dec_out[:, -1, :]).float() / max(temperature, 1e-5)
        if top_k is not None and 0 < top_k < logits.size(-1):
            vals, _ = torch.topk(logits, top_k)
            logits = logits.masked_fill(logits < vals[:, [-1]], float("-inf"))
        if top_p is not None and top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            probs = torch.softmax(sorted_logits, dim=-1)
            cumulative = torch.cumsum(probs, dim=-1)
            remove = cumulative > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False
            sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, sorted_idx, sorted_logits)
            logits = filtered
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        decoder_ids = torch.cat([decoder_ids, next_id], dim=1)
        if int(next_id.item()) == EOS_ID:
            break

    return tokenizer.decode(decoder_ids[0, 1:].tolist()).strip()

# Load the best checkpoint if available.
if os.path.exists(BEST_CKPT):
    best = torch.load(BEST_CKPT, map_location=DEVICE)
    model.load_state_dict(best["model_state_dict"])
    print("Loaded best checkpoint; step =", best.get("global_step"))
elif os.path.exists(LATEST_CKPT):
    latest = torch.load(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(latest["model_state_dict"])
    print("Loaded latest checkpoint; step =", latest.get("global_step"))

prompt, reference = val_examples[0]
print("PROMPT:", prompt[:500], "...")
print("\nGENERATED:", generate_continuation(model, tokenizer, prompt))
print("\nREFERENCE:", reference[:500], "...")

Loaded best checkpoint; step = 11200
PROMPT: Appellate Tribunal For Electricity M/S Supreme Tex Mart Ltd vs Punjab State Electricity Board & Ors on 27 April, 2012 COURT - I Before the Appellate Tribunal for Electricity ( Appellate Jurisdiction ) IA No. 157 of 2012 in Dated : 27th April, 2012 Present : Hon'ble Mr. Justice M. Karpaga Vinayagam, Chairperson Hon'ble Mr. Rakesh Nath, Technical Member M/s Supreme Tex Mart Ltd. .....Appellant(s) Versus Punjab State Electricity Board & Ors. ....Respondent(s) Counsel for the Appellant(s): Mr. Lalit ...

GENERATED: that the Appellate Tribunal is within a period of three weeks from the date of receipt of a copy of this order . 2 . The appeal is dismissed as withdrawn . ( Dr . Satyagopal Korlapati ) ( Dr . Sharma ) Member ( Technical ) Member ( Technical )

REFERENCE: permission to withdraw the Application. Permission is granted. Accordingly, the Application is dismissed as withdrawn. Consequently, the Appeal is also dismissed. (Rakesh Nath) (Just

## 10. Validation generation metrics (optional)

In [ ]:
# Install only if you want these text-overlap/semantic metrics:
# !pip -q install bert-score rouge-score sacrebleu

N_EVAL = min(50, len(val_examples))
hyps, refs = [], []
for prompt, reference in val_examples[:N_EVAL]:
    hyps.append(generate_continuation(model, tokenizer, prompt, max_new_tokens=MAX_DEC_LEN))
    refs.append(reference)

try:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = np.mean([scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(refs, hyps)])
    print(f"ROUGE-L F1: {rouge_l:.4f}")
except ImportError:
    print("Install rouge-score to calculate ROUGE-L.")

try:
    import sacrebleu
    print(f"BLEU-4: {sacrebleu.corpus_bleu(hyps, [refs]).score:.2f}")
except ImportError:
    print("Install sacrebleu to calculate BLEU.")

## 11. Generate official Phase-1 predictions

In [ ]:
# Expected test file: DATA_DIR/Test/Phase1-Test.json.
# Output schema: {test_id: {"text": "generated continuation"}}
import json

test_path = os.path.join('content', "Test", "Phase1-Test.json")
if not os.path.exists(test_path):
    try:
        from huggingface_hub import hf_hub_download
        test_path = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            filename="Phase_1_Pretraining/Test/test.json",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )
    except Exception as exc:
        raise FileNotFoundError("Set DATA_DIR to the Phase-1 data folder or configure HF_TOKEN.") from exc

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

predictions = {}
for test_id, item in test_data.items():
    prompt = item.get("text", "") if isinstance(item, dict) else str(item)
    predictions[str(test_id)] = {"text": generate_continuation(model, tokenizer, prompt)}

submission_dir = os.path.join(WORK_DIR, "submission")
os.makedirs(submission_dir, exist_ok=True)
prediction_path = os.path.join(submission_dir, "prediction.json")
with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

import zipfile
zip_path = os.path.join(submission_dir, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")
print(f"Wrote {prediction_path}")
print(f"Wrote {zip_path} with prediction.json at ZIP root")
print("Predictions:", len(predictions))

Wrote /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/submission/prediction.json
Wrote /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/submission/submission.zip with prediction.json at ZIP root
Predictions: 1000


## Implementation notes / checklist

- **Tokenizer compatibility:** custom `BPETokenizer` from the Transformer-XL notebook, saved as `bpe_legal_16384.json`. Keep this JSON with the model checkpoint.
- **No train/validation leakage:** split documents before tokenizer training; tokenizer is trained on training documents only.
- **Correct decoder shift:** decoder input is `[BOS] + target[:-1]`; labels are the unshifted target sequence, including EOS.
- **Validation:** loss is total negative log-likelihood divided by the number of non-padding target tokens. Perplexity is `exp(validation token loss)`.
- **Resume:** rerunning the training cell loads `encoder_decoder_latest.pt` if present. Delete that file to start a fresh run.
- **Memory:** begin with the default 512/128 sequence lengths, batch size 2, and gradient accumulation 8. Reduce lengths or batch size if you hit CUDA OOM.
- **Scaling:** after a stable smoke test, increase `MAX_TRAIN_DOCS`, then train for longer. Model width/depth can be scaled after measuring GPU memory and throughput.

- **Data pipeline parity:** downloads the same four JSON sources, uses the Transformer-XL reservoir-sampling caps (20k acts, 300k high courts, 50k supreme court, 200k tribunals), concatenates/shuffles them, then applies the same normalization, English filtering, exact deduplication, and MinHash-LSH near-deduplication.
- **Credential hygiene:** reads `HF_TOKEN` from the environment rather than embedding a Hugging Face access token in the notebook.
